# SWOT Canal Observability — Confidence Level (CL) Computation

This notebook computes **CL**, the observability confidence level for each
canal segment, from the pipeline output produced by `dem_sample.py`.

## What is CL?

CL quantifies how reliably the SWOT satellite can observe water surface elevation
(WSE) over an individual irrigation canal segment. It combines three independent
dimensions into a single score in [0, 1]:

| Component | Variable | What it measures |
|-----------|----------|------------------|
| Physical realism | `s_physical_realism` | Does the SWOT-derived slope direction and magnitude agree with the DEM terrain? (Direction agreement is a continuous confidence, not a binary same-sign/opposite-sign flag — see Cell 4.) |
| Statistical robustness | `s_statistical_robustness` | Is the slope well-constrained (high SNR, low residual noise)? |
| Spatial coherence | `s_spatial_coherence` | Does SWOT observe most of the canal length continuously? |

The formula is:
```
CL_geomean = (s_physical_realism × s_statistical_robustness × s_spatial_coherence)^(1/3)
CL = CL_geomean × min(s_physical_realism, s_statistical_robustness, s_spatial_coherence)^0.5
```

The minimum-dimension penalty ensures that a canal must score reasonably well on
ALL three dimensions to receive a high confidence level.

## Class thresholds

Derived from a Gaussian kernel density estimate of the CL distribution (density
valleys nearest the low and high ends of the distribution), checked for stability
across kernel bandwidths 0.10-0.20.

| Class | CL range | Interpretation |
|-------|-----------|----------------|
| Low | 0.000 – 0.108 | SWOT rarely produces usable observations |
| Moderate | 0.108 – 0.597 | Observations available but with caveats |
| High | 0.597 – 1.000 | SWOT reliably observes this canal |

Canals without a DEM slope have `CL = NaN`; see Cell 7 for how these are
labelled in the `class` column.

## Input required

Set `PIPELINE_OUTPUT_CSV` in Cell 1 to the path of
`wse_dem_slope_comparison.csv` produced by `dem_sample.py` for your region.


In [ ]:
# ── Cell 1: Configuration ─────────────────────────────────────────────────────
from pathlib import Path

# Path to the pipeline output CSV produced by dem_sample.py.
# Example: "/data/runs/iran_from_0_to_5000/metrics/wse_dem_slope_comparison.csv"
PIPELINE_OUTPUT_CSV = Path("/path/to/your/run/metrics/wse_dem_slope_comparison.csv")

# Where to write the final confidence-level CSV.
# Defaults to the same directory as the input file.
OUTPUT_CSV = PIPELINE_OUTPUT_CSV.parent / "confidence_levels_CL.csv"

# ── Tunable parameters ────────────────────────────────────────────────────────

# Exponent for the minimum-dimension penalty in CL.
# GAMMA=0.5 (default) is a moderate penalty; increase for stricter penalisation.
GAMMA = 0.5

# Weight of slope-direction vs. slope-magnitude agreement in s_physical_realism.
# ALPHA=0.7 means direction agreement contributes 70%.
ALPHA = 0.7

# Weight of coverage vs. contiguity in s_spatial_coherence.
# BETA=0.7 means coverage contributes 70%.
BETA = 0.7

# Winsorized-quantile anchors [low_q, high_q] for each raw metric.
# See winsorized_score() in Cell 2 for how these are applied.
SNR_ANCHOR     = (0.01, 0.50)   # wse_slope_snr, higher is better
DISP_ANCHOR    = (0.10, 0.981)  # wse_noise_sigma_m, lower is better
COV_ANCHOR     = (0.01, 0.50)   # obs_coverage_frac, higher is better
CONTIG_ANCHOR  = (0.01, 0.50)   # obs_contiguity_frac, higher is better
MAG_ANCHOR     = (0.10, 0.75)   # slope_magnitude_diff_m_per_m, lower is better

EPS = 1e-9   # small constant to avoid issues when components are exactly zero

print(f"Input : {PIPELINE_OUTPUT_CSV}")
print(f"Output: {OUTPUT_CSV}")


In [ ]:
# ── Cell 2: Imports & helpers ─────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression


def winsorized_score(series: pd.Series, low_q: float, high_q: float,
                     higher_is_better: bool = True) -> pd.Series:
    """
    Map a metric series to [0, 1] via winsorized linear scaling, anchored at
    the low_q/high_q percentiles of the series itself.
    """
    x     = pd.to_numeric(series, errors="coerce")
    valid = x.dropna()
    if len(valid) == 0:
        return pd.Series(np.nan, index=series.index)
    p_low  = float(np.quantile(valid, low_q))
    p_high = float(np.quantile(valid, high_q))
    if p_high == p_low:
        return pd.Series(0.5, index=series.index)
    raw = (x - p_low) / (p_high - p_low)
    if not higher_is_better:
        raw = 1.0 - raw
    return raw.clip(0.0, 1.0)


def score_slope_direction_isotonic(wse_slope: pd.Series, dem_slope: pd.Series) -> pd.Series:
    """
    Continuous slope-direction confidence score.

    Rather than a binary same-sign/opposite-sign flag, this calibrates a
    confidence value from the empirical sign-disagreement rate as a function
    of the smaller of the two slope magnitudes (on a log scale), via
    monotonic (isotonic) regression: disagreement should only get LESS
    likely as the smaller slope magnitude grows (a small, noise-dominated
    slope is inherently more likely to disagree in sign with its DEM
    counterpart than a large, well-resolved one).

    score = 0.5 + 0.5 * confidence  if signs agree
    score = 0.5 - 0.5 * confidence  if signs disagree
    where confidence = clip(1 - 2 * p_hat, 0, 1) and p_hat is the isotonic
    fit of P(disagree | log10(min(|wse_slope|, |dem_slope|))).

    NaN where either slope is missing.
    """
    valid = wse_slope.notna() & dem_slope.notna()
    out = pd.Series(np.nan, index=wse_slope.index)
    if valid.sum() == 0:
        return out

    w = wse_slope[valid].values
    d = dem_slope[valid].values
    signs_agree = (np.sign(w) == np.sign(d)).astype(int)
    min_mag = np.clip(np.minimum(np.abs(w), np.abs(d)), 1e-12, None)
    log_min_mag = np.log10(min_mag)
    disagree = 1 - signs_agree

    iso = IsotonicRegression(increasing=False, out_of_bounds="clip")
    p_hat = iso.fit_transform(log_min_mag, disagree)
    confidence = np.clip(1 - 2 * p_hat, 0, 1)
    scores = np.where(signs_agree == 1, 0.5 + 0.5 * confidence, 0.5 - 0.5 * confidence)
    out.loc[valid] = scores
    return out


print("Helpers ready.")


In [ ]:
# ── Cell 3: Load pipeline output ──────────────────────────────────────────────
if not PIPELINE_OUTPUT_CSV.exists():
    raise FileNotFoundError(
        f"Pipeline output not found: {PIPELINE_OUTPUT_CSV}\n"
        "Run the full pipeline first (snakemake --cores N)."
    )

df = pd.read_csv(PIPELINE_OUTPUT_CSV, low_memory=False)

# Coerce all metric columns to numeric
METRIC_COLS = [
    "wse_slope_m_per_m", "wse_slope_abs_m_per_m", "wse_noise_sigma_m",
    "canal_length_m", "obs_coverage_frac", "obs_contiguity_frac",
    "wse_slope_snr", "dem_slope_m_per_m",
]
for c in METRIC_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Ensure wse_slope_abs_m_per_m is always the absolute value of the WSE slope
df["wse_slope_abs_m_per_m"] = df["wse_slope_abs_m_per_m"].fillna(df["wse_slope_m_per_m"].abs())

has_dem = df["dem_slope_m_per_m"].notna()

print(f"Loaded {len(df):,} canals")
print(f"Canals with DEM slope (needed for CL): {has_dem.sum():,}")
df[METRIC_COLS].describe().round(4)

In [ ]:
# ── Cell 4: Individual metric scores (winsorized, all in [0, 1]) ──────────────
#
# Quantile anchors are derived from the loaded dataset (set in Cell 1), so
# they adapt to the metric distributions of each region.

# Statistical robustness
df["score_snr"]        = winsorized_score(df["wse_slope_snr"], *SNR_ANCHOR, higher_is_better=True)
df["score_dispersion"] = winsorized_score(df["wse_noise_sigma_m"], *DISP_ANCHOR, higher_is_better=False)

# Spatial coherence
df["score_coverage"]    = winsorized_score(df["obs_coverage_frac"], *COV_ANCHOR, higher_is_better=True)
df["score_contiguity"]  = winsorized_score(df["obs_contiguity_frac"], *CONTIG_ANCHOR, higher_is_better=True)

# Physical realism — slope direction agreement between SWOT and DEM
# Continuous isotonic-regression confidence (see Cell 2); NaN if either slope is missing.
df["score_slope_direction"] = score_slope_direction_isotonic(
    df["wse_slope_m_per_m"], df["dem_slope_m_per_m"]
)

# Physical realism — slope magnitude consistency
df["slope_magnitude_diff_m_per_m"] = (
    df["wse_slope_abs_m_per_m"] - df["dem_slope_m_per_m"].abs()
).abs()
df["score_slope_magnitude"] = winsorized_score(
    df["slope_magnitude_diff_m_per_m"], *MAG_ANCHOR, higher_is_better=False
)

score_cols = [
    "score_snr", "score_dispersion",
    "score_coverage", "score_contiguity",
    "score_slope_direction", "score_slope_magnitude",
]
print("Individual score statistics:")
df[score_cols].describe().round(3)


In [ ]:
# ── Cell 5: Three composite component scores ──────────────────────────────────
#
# s_physical_realism
#   Weighted geometric mean of slope-direction and slope-magnitude agreement.
#   Requires DEM data; NaN for canals without a DEM slope.
#
# s_statistical_robustness
#   Geometric mean of SNR and dispersion (noise) scores.
#
# s_spatial_coherence
#   Weighted geometric mean of spatial coverage and contiguity
#   (coverage weighted more heavily, BETA=0.7 by default).

df["s_physical_realism"] = (
    (df["score_slope_direction"] + EPS) ** ALPHA *
    (df["score_slope_magnitude"] + EPS) ** (1 - ALPHA)
)
df.loc[
    df["score_slope_direction"].isna() | df["score_slope_magnitude"].isna(),
    "s_physical_realism"
] = np.nan

df["s_statistical_robustness"] = np.sqrt(
    (df["score_snr"] + EPS) * (df["score_dispersion"] + EPS)
)

df["s_spatial_coherence"] = (
    (df["score_coverage"]   + EPS) ** BETA *
    (df["score_contiguity"] + EPS) ** (1 - BETA)
)

has_all = df["s_physical_realism"].notna()
print(f"Canals with all three components available: {has_all.sum():,} / {len(df):,}")
print("(Canals without DEM slope will have CL = NaN)")
print("\nComponent statistics (rows with DEM):")
df.loc[has_all, ["s_physical_realism", "s_statistical_robustness", "s_spatial_coherence"]].describe().round(3)


In [ ]:
# ── Cell 6: Compute CL ─────────────────────────────────────────────────────
#
# CL_geomean = (s_physical_realism × s_statistical_robustness × s_spatial_coherence)^(1/3)
#   Geometric mean: balanced summary across all three dimensions.
#
# CL = CL_geomean × min(s_physical_realism, s_statistical_robustness, s_spatial_coherence)^GAMMA
#   Extra penalty for whichever dimension is weakest, so a canal must perform
#   reasonably well on ALL three to receive a high confidence level.

comp_min = df[["s_physical_realism", "s_statistical_robustness", "s_spatial_coherence"]].min(axis=1)

df["CL_geomean"] = np.where(
    has_all,
    (
        (df["s_physical_realism"]       + EPS) *
        (df["s_statistical_robustness"] + EPS) *
        (df["s_spatial_coherence"]      + EPS)
    ) ** (1/3),
    np.nan
)
df["CL_geomean"] = df["CL_geomean"].clip(0, 1)

df["CL"] = np.where(
    has_all,
    df["CL_geomean"] * (comp_min ** GAMMA),
    np.nan
)
df["CL"] = df["CL"].clip(0, 1)

print("CL distribution (canals with DEM slope):")
df["CL"].describe().round(3)


In [ ]:
# ── Cell 7: Class distribution ────────────────────────────────────────────────
#
# Canals without a DEM slope (CL = NaN) are assigned "Moderate" by convention
# rather than left unclassified. This is a placeholder, not a genuine
# classification -- filter on df["CL"].notna() first if you need only
# canals with a real confidence assessment.

BINS       = [0.0, 0.108, 0.597, 1.0000001]
BIN_LABELS = ["Low", "Moderate", "High"]

df["class"] = "Moderate"
classified = pd.cut(df["CL"].dropna(), bins=BINS, labels=BIN_LABELS, right=False)
df.loc[classified.index, "class"] = classified.astype(str)

counts = df.loc[df["CL"].notna(), "class"].value_counts().reindex(BIN_LABELS, fill_value=0)
pcts   = (counts / counts.sum() * 100).round(1)

print("CL class distribution (canals with a real CL only):")
for label in BIN_LABELS:
    print(f"  {label:>10s}: {counts[label]:>6,} canals  ({pcts[label]:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Quantile curve
vals = df["CL"].dropna().sort_values().values
q    = np.linspace(0, 1, len(vals))
axes[0].plot(q, vals, lw=1.5, color="steelblue")
axes[0].axhline(0.108, ls="--", lw=0.9, color="orange", label="0.108 (low/mod)")
axes[0].axhline(0.597, ls="--", lw=0.9, color="green",  label="0.597 (mod/high)")
axes[0].set_xlabel("Quantile")
axes[0].set_ylabel("CL")
axes[0].set_title("CL quantile curve")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.4)

# Bar chart of class distribution
colors = ["#d62728", "#ff7f0e", "#2ca02c"]
axes[1].bar(BIN_LABELS, pcts.values, color=colors, edgecolor="white")
axes[1].set_ylabel("% of canals")
axes[1].set_title("CL class distribution")
axes[1].grid(axis="y", alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 8: Component score distributions ────────────────────────────────────
# A component that is systematically low for your region may indicate
# a data quality issue worth investigating.

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
component_labels = {
    "s_physical_realism":       "Physical realism (s_physical_realism)",
    "s_statistical_robustness": "Statistical robustness (s_statistical_robustness)",
    "s_spatial_coherence":      "Spatial coherence (s_spatial_coherence)",
}

for ax, (col, label) in zip(axes, component_labels.items()):
    v = df[col].dropna().sort_values().values
    q = np.linspace(0, 1, len(v))
    ax.plot(q, v, lw=1.5)
    ax.set_title(label, fontsize=9)
    ax.set_xlabel("Quantile")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.4)

axes[0].set_ylabel("Score")
fig.suptitle("Component score distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 9: Save output ───────────────────────────────────────────────────────
SAVE_COLS = [
    "grain_id",
    "canal_length_m",
    "wse_slope_m_per_m", "wse_slope_abs_m_per_m",
    "wse_noise_sigma_m",
    "obs_coverage_frac", "obs_contiguity_frac",
    "wse_slope_snr",
    "dem_slope_m_per_m",
    "slope_magnitude_diff_m_per_m",
    "score_slope_direction", "score_slope_magnitude",
    "score_snr", "score_dispersion",
    "score_coverage", "score_contiguity",
    "s_physical_realism", "s_statistical_robustness", "s_spatial_coherence",
    "CL", "class",
]

output_df = df[[c for c in SAVE_COLS if c in df.columns]].copy()
output_df.to_csv(OUTPUT_CSV, index=False)

print(f"[OK] Saved: {OUTPUT_CSV}")
print(f"     Total canals : {len(output_df):,}")
print(f"     With CL      : {output_df['CL'].notna().sum():,}")
